In [ ]:
import os 
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

def dashscope_embedding(text:str,model:str=None):
    if model ==None:
        model ="text-embedding-v4"
    response = client.embeddings.create(
        input = text,
        model=model
    )
    return response

response = dashscope_embedding(text='要生成embedding的输入文本,字符串形式')

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader('/root/ai_rag_project/data_base/pumpkin_book.pdf')

pdf_pages = loader.load()


In [ ]:
print(f'载入后的变量类型为：{type(pdf_pages)}, 该PDF一共包含{len(pdf_pages)}页')

In [ ]:
pdf_page = pdf_pages[1]  # 取出第二页

# 修正变量名和括号结构
print(f"每一个元素的类型：{type(pdf_page)}")
print(f"该文档的描述性数据：{pdf_page.metadata}")
print("-" * 30) # 分割线
print(f"查看该文档的内容:\n{pdf_page.page_content[:500]}") # 先看前500字

In [ ]:
import re
pattern = re.compile(r'[^\u4e00-\u9fff](\n)[^\u4e00-\u9fff]', re.DOTALL)
pdf_page.page_content = re.sub(pattern, lambda match: match.group(0).replace('\n', ''), pdf_page.page_content)

pdf_page.page_content = pdf_page.page_content.replace('•', '')
pdf_page.page_content = pdf_page.page_content.replace(' ', '')
print(pdf_page.page_content)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
OVERLAP_SIZE=50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE
)
text_splitter.split_text(pdf_page.page_content[0:1000])
split_docs =text_splitter.split_documents(pdf_pages)
print(f'切分后的文件数量：{len(split_docs)}')
print(f'切分后的字符数可以大致评估的token数：{sum([len(doc.page_content)for doc in split_docs])}')

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
embeddings =DashScopeEmbeddings(
    model='text-embedding-v4',
    dashscope_api_key=os.environ['DASHSCOPE_API_KEY']
)


In [ ]:
from langchain_chroma import Chroma
vectordb=Chroma(
    collection_name='pumpkin_collection',
    embedding_function=embeddings,
    persist_directory="/root/ai_rag_project/data_base/chroma_langchain_db"
)
